**Cell #01**

# RAG11 — Stage 2 (Yoga-Sūtra): Ask Smart Questions, Some in Devanagari

Runs five hand-picked questions about the **Yoga-Sūtra of Patañjali and the Yoga-Bhāṣya** through the same
retrieval + generation pipeline as the nutrition notebooks, but against the one source that holds them:
Michel Angot's French edition (`Yogasutra.janvier. 2020.pdf, éd. 2021.pdf`, chunked by
`stage1_1_eda_packages/source18_yoga_sutra_angot.py`).

**Speaking language.** `speaking_language = "EN"` (cell #02): questions may be in *any* language or script
(English, French, Devanagari...), and every answer is written in English. Two things make that work smoothly:

1. **Question understanding** (`prepare_question()`, one small Claude call before retrieval): detects the language,
   translates the question into English, and rewrites it as a neutral **search query** in the corpus's language
   (Sanskrit terms in IAST). Retrieval searches with that clean query; the answering model sees the original
   question plus its English translation and IAST rendering.
2. **An enforced answer language** (`ask_question(answer_language=...)`): an explicit instruction in the system
   prompt, so the answer is English even when the question is Hindi and the excerpts are French.

What is different from the nutrition notebooks:

- **One source only.** `ask_question(filter_owner=...)` restricts every retrieval step to this book's `rowGUID`.
- **A Yoga-specific system prompt** (`ask_question(system_prompt=...)`).
- **No parent expansion.** A "parent" here is one whole sūtra with its Bhāṣya and Angot's notes (often 10 000+
  characters); the 400-token child chunks are the precise units, so `expand_to_parents=False`.
- **Shared output design.** `show_qa()` from `reusable_code/display.py` renders the Question / Answer cards
  (also used by `stage2_ask_examples8_nutriciology.ipynb`).

**Before running this notebook**: `stage1_1` must have chunked this book with
`MAX_NUMBER_OF_PAGES_TO_USE = None` (with the default cap of 100 pages the sūtra text itself, which starts on
page 244, is empty) and `stage1_2` must have loaded it. Cell #04 checks this and tells you what is missing.

In [ ]:
# Cell #02
from reusable_code import (
    init_clients, ask_question, prepare_question, show_qa, show_summary,
    contains_devanagari, romanize_devanagari, language_name, GENERATION_MODEL,
)
from reusable_code.env import optional_env

# Every answer is written in this language, whatever language/script the question is in
# (ISO 639-1 code: EN, FR, DE, ES, HI, ...). The default comes from SPEAKING_LANGUAGE in .env.
speaking_language = "EN"

clients = init_clients()
print("Clients ready. Supabase project:", optional_env("PUBLIC_SUPABASE_URL"), "| model:", GENERATION_MODEL)
print("Speaking language:", speaking_language, f"({language_name(speaking_language)})")

**Cell #03**

## Find the book in the database and check it is fully loaded

Looks the source up by file name (its `source_key` can change if files are added to the Drive folder),
then counts its child chunks and its per-sūtra parent sections. The full book has 195 sūtra sections.

In [ ]:
# Cell #04
ys_rows = (clients.supabase.table("rag11_data_sources").select('"rowGUID",source_key,filename')
           .ilike("filename", "%Yogasutra%").execute().data)
if not ys_rows:
    raise RuntimeError("The Yoga-Sutra book is not in rag11_data_sources -- run stage1_1 and stage1_2 first.")

YS_OWNER_GUID = ys_rows[0]["rowGUID"]

n_children = (clients.supabase.table("rag11_chunks_child_table").select('"rowGUID"', count="exact")
              .eq("rowOwnerGUID", YS_OWNER_GUID).limit(1).execute().count)
n_sutra_parents = (clients.supabase.table("rag11_chunks_parent_table").select('"rowGUID"', count="exact")
                   .eq("rowOwnerGUID", YS_OWNER_GUID).like("title", "Yoga-S%tra%").limit(1).execute().count)

print(f"{ys_rows[0]['source_key']}: {n_children} child chunks, {n_sutra_parents} of 195 sutra sections loaded")
if n_sutra_parents < 195:
    print("\nWARNING: the sutra text is not (fully) loaded, so questions about individual sutras will get "
          "'the excerpts do not contain...' answers.\n"
          "  1. stage1_1_extract_and_chunk.ipynb: set MAX_NUMBER_OF_PAGES_TO_USE = None and re-run\n"
          "  2. stage1_2_eda_load_chunks.ipynb: re-run (only new/changed chunks are embedded)\n"
          "  3. stage1_9_eda_verify_all_data.ipynb: confirm PASS")

**Cell #05**

## The Yoga-specific system prompt and the pipeline

Same contract as the nutrition prompt (answer only from the numbered excerpts; a `Short answer: Yes|No` line
only for genuine yes/no questions) plus what a reader of *this* book needs: who is speaking (sūtra, Bhāṣya, or
Angot) and sūtra references. The **answer language is not written into this prompt**: `answer_language=` appends
it, so the same prompt works for any `speaking_language`.

`ask_ys()` takes an already prepared question (see `prepare_question()`) and runs hybrid search + multi-query
splitting + reranking over this book only.

In [ ]:
# Cell #06
YS_SYSTEM_PROMPT = """You are a research assistant for a French scholarly edition of the Yoga-Sutra of \
Patanjali and the Yoga-Bhasya of Vyasa (Michel Angot, 3rd edition 2021). Answer strictly using the numbered \
excerpts in the user message -- do not rely on outside knowledge, and say plainly if the excerpts don't \
contain enough information to answer.

The excerpts are mostly French; Sanskrit appears in IAST transliteration. The question may be written in any \
language or script; lines such as "[English: ...]" or "[IAST: ...]" after it are its translation and \
transliteration, added to help you.

Whenever the excerpts allow it: give the sutra reference (e.g. II.35), quote the key Sanskrit term in IAST, \
and say whether a claim comes from the Sutra itself, from the Bhasya, or from Angot's own commentary.

First decide whether the question is a Yes/No question, i.e. it is worded "is/does/can/are ... ?" (or "क्या ...?") and \
a plain yes or no truly answers it. Questions that ask "what", "how", "why", "who", "which", or "what happens" are \
NEVER Yes/No questions, however the excerpts turn out.
  - If it is a Yes/No question, begin your reply with exactly this one line:
        Short answer: Yes
    or
        Short answer: No
    then a blank line, then the full explanation.
  - Otherwise skip the "Short answer" line and give the full explanation.
  - If the excerpts do not contain enough to answer, never write a "Short answer" line: say what is missing.

Keep the explanation grounded in the excerpts."""

YS_CORPUS_HINT = ("a French scholarly edition of the Yoga-Sutra and Yoga-Bhasya (Michel Angot) whose text is "
                  "French prose with Sanskrit terms in IAST transliteration")


def ask_ys(prepared, **overrides) -> dict:
    """Run one prepared question (a PreparedQuestion) through the Yoga-Sutra pipeline."""
    options = dict(
        match_count=6,
        use_hybrid=True,            # dense + keyword: exact sutra terms (IAST) are found by the keyword half
        use_multi_query=True,       # split compound questions into sub-questions
        use_hyde=False,
        use_rerank=True,            # cross-encoder re-scores the wide candidate pool
        expand_to_parents=False,    # a parent here is a whole sutra + commentary; keep the precise child chunks
        filter_owner=YS_OWNER_GUID,
        system_prompt=YS_SYSTEM_PROMPT,
        answer_language=speaking_language,
        retrieval_query=prepared.retrieval_query,
    )
    options.update(overrides)
    return ask_question(prepared.llm_question, **options)

**Cell #07**

## The five questions

Each targets a different part of the book and a different retrieval difficulty:

1. **Devanagari sūtra + a "how does the Bhāṣya explain" question**: the sūtra (I.2) is quoted in Devanagari, so
   only its IAST rendering can reach the chunk; the answer needs both the sūtra and the Bhāṣya's gloss of *nirodha*.
2. **Interpretive question about Angot's method** (English): the answer lives in one of his appendix notices
   (*Adhikāra I.1 et les destinataires du Yoga-Sūtra*), written in French, so this is cross-lingual retrieval.
3. **Devanagari sūtra + a consequence** (II.35): quote in Devanagari, ask what follows from it; the Bhāṣya and
   Angot's notes add nuance beyond the one-line sūtra.
4. **A multi-part comparison** (English, IAST terms): the five *vṛtti*-s and which are *kliṣṭa* / *akliṣṭa*
   (I.5-11): spread across seven sūtras, the case multi-query splitting exists for.
5. **A Hindi yes/no question written entirely in Devanagari**: *is Īśvara in the Yoga-Sūtra a creator god?* A
   clean Yes/No with an important nuance (I.23-26, II.1, II.45), and no Latin letters at all in the question.
   The answer still comes back in English.

In [ ]:
# Cell #08
YS_QUESTIONS = [
    # 1. Devanagari sutra (I.2) + English question about the Bhasya's gloss of nirodha.
    "योगश्चित्तवृत्तिनिरोधः -- what does Yoga-Sūtra I.2 define yoga as, and how does the Bhāṣya explain the word nirodha?",
    # 2. Interpretive: Angot's reading of the very first word, answer is in a French appendix notice.
    "Why does Angot read the opening word 'atha' of Yoga-Sūtra I.1 as an adhikāra, and who are the intended addressees of the text?",
    # 3. Devanagari sutra (II.35) + what follows from it.
    "अहिंसाप्रतिष्ठायां तत्सन्निधौ वैरत्यागः -- according to II.35, what happens in the presence of someone firmly established in ahiṃsā?",
    # 4. Multi-part comparison across I.5-I.11 -- the case for multi-query splitting.
    "How do the five vṛtti-s (pramāṇa, viparyaya, vikalpa, nidrā, smṛti) differ from one another, and which of them can be kliṣṭa or akliṣṭa?",
    # 5. Hindi, all Devanagari, clean Yes/No with a nuance.
    "क्या योगसूत्र में ईश्वर जगत् का सृष्टिकर्ता है?",
]

**Cell #09**

## Step 1 — what the questions turn into

`prepare_question()` runs once per question (one small Claude call each). The table shows what it understood:
the detected language, the English translation, and the neutral **search query** that retrieval will use instead of
the raw text. If that call ever fails, the original question (plus an IAST rendering of any Devanagari) is used, so
a question is never lost.

In [ ]:
# Cell #10
ys_prepared = [prepare_question(q, language=speaking_language, corpus_hint=YS_CORPUS_HINT) for q in YS_QUESTIONS]

show_summary(
    [(i, p.language or "?", p.translation or "(same)", p.retrieval_query) for i, p in enumerate(ys_prepared, start=1)],
    ["#", "language", "understood as", "search query used for retrieval"],
)

**Cell #11**

## Step 2 — run all 5 questions

Each prepared question goes through `ask_ys()` and is displayed as a **Question / Answer** card by `show_qa()`
(`reusable_code/display.py`): the answer's Markdown is rendered as paragraphs and lists, a genuine yes/no shows as a
badge, and the card has `height: auto`, so the full answer is always visible. The footer shows the pipeline's
bookkeeping: excerpts kept out of the candidates the reranker saw, compressed page ranges, and the sub-questions.

(If your Jupyter front end puts a long output in a scrolling box, right-click it and choose "Disable Scrolling for
Outputs"; JupyterLab and PyCharm never do.)

In [ ]:
# Cell #12
ys_results = []
for number, (question, prepared) in enumerate(zip(YS_QUESTIONS, ys_prepared), start=1):
    result = ask_ys(prepared)
    ys_results.append(result)
    show_qa(number, question, result, prepared)

**Cell #13**

## Summary table

In [ ]:
# Cell #14
show_summary(
    [
        (i, "Devanagari" if contains_devanagari(q) else "Latin", p.language or "?", r["short_answer"] or "n/a",
         len(r["subquestions"] or []), r["chunks_used"], q[:80] + ("..." if len(q) > 80 else ""))
        for i, (q, p, r) in enumerate(zip(YS_QUESTIONS, ys_prepared, ys_results), start=1)
    ],
    ["#", "script", "question language", "short answer", "sub-Qs", "excerpts", "question"],
)

**Cell #15**

## What does question understanding buy? (retrieval only)

Compares **hybrid retrieval alone** for question 3, once with the raw Devanagari question and once with the search
query from step 1, showing which half of the hybrid search (dense or keyword) found each chunk. The raw Devanagari
has no letters in common with any chunk, so only the search query can hit the II.35 section once the sūtra text is
fully loaded.

In [ ]:
# Cell #16
from reusable_code import hybrid_search

for label, text in [("raw question", YS_QUESTIONS[2]), ("search query", ys_prepared[2].retrieval_query)]:
    rows = hybrid_search(text, match_count=5, filter_owner=YS_OWNER_GUID)
    print(f"--- {label}: {text[:100]}")
    for r in rows:
        first_line = r["rowJSON"]["text"].split("\n", 1)[0][:95]
        print(f"  dense#{r['dense_rank']!s:<4} keyword#{r['keyword_rank']!s:<5} {first_line}")

**Cell #17**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock recovery and conflict resolution).

In [ ]:
# Cell #18
from reusable_code import save_to_github

save_to_github("stage2_ask_examples7_ys.ipynb - Yoga-Sutra questions, speaking_language, shared card design")